# CascadeTorch: Results Visualization
Visualize spike rate predictions from pretrained CASCADE models.

In [ ]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio

# Set plot defaults
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

from cascade2p import cascade
from src.utils.noise import calculate_noise_levels

## 1. Load Example Data
Load Allen Brain Observatory calcium imaging data.

In [ ]:
example_file = os.path.join(project_root, 'Example_datasets', 
    'Allen-Brain-Observatory-Visual-Coding-30Hz', 
    'Experiment_552195520_excerpt.mat')

traces = sio.loadmat(example_file)['dF_traces']
frame_rate = 30  # Hz

print(f"Loaded {traces.shape[0]} neurons, {traces.shape[1]} timepoints")
print(f"Frame rate: {frame_rate} Hz")
print(f"Duration: {traces.shape[1]/frame_rate:.1f} seconds")

## 2. Noise Level Distribution

In [ ]:
noise_levels = calculate_noise_levels(traces, frame_rate)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(noise_levels, bins=30, edgecolor='black', alpha=0.7)
ax.set_xlabel('Noise level (% s^(1/2))')
ax.set_ylabel('Number of neurons')
ax.set_title(f'Noise level distribution (n={len(noise_levels)} neurons)')
ax.axvline(np.median(noise_levels), color='red', linestyle='--', label=f'Median: {np.median(noise_levels):.2f}')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Run Inference with Pretrained Model

In [ ]:
model_name = 'Global_EXC_30Hz_smoothing200ms'

# Download model if not present
model_path = os.path.join(project_root, 'Pretrained_models', model_name)
if not os.path.exists(model_path):
    cascade.download_model(model_name, 
                          model_folder=os.path.join(project_root, 'Pretrained_models'))

# Run inference
os.chdir(project_root)
spike_rates = cascade.predict(model_name, traces / 100, 
                              model_folder=os.path.join(project_root, 'Pretrained_models'))
print(f"Predictions shape: {spike_rates.shape}")

## 4. Visualize Predictions

In [ ]:
# Select 6 random neurons to display
np.random.seed(42)
neuron_indices = np.random.choice(traces.shape[0], size=6, replace=False)

fig, axes = plt.subplots(3, 2, figsize=(16, 10), sharex=True)
time = np.arange(traces.shape[1]) / frame_rate

for idx, (ax, neuron) in enumerate(zip(axes.flat, neuron_indices)):
    # Plot dF/F trace
    ax.plot(time, traces[neuron, :] / 100, color='steelblue', alpha=0.8, label='dF/F')
    
    # Plot spike rate prediction (shifted down for visibility)
    ax.plot(time, spike_rates[neuron, :] - 0.5, color='orangered', alpha=0.8, label='Spike rate')
    
    ax.set_ylabel(f'Neuron {neuron}')
    ax.set_xlim(1, min(50, traces.shape[1] / frame_rate - 1))
    if idx == 0:
        ax.legend(loc='upper right')

axes[-1, 0].set_xlabel('Time (s)')
axes[-1, 1].set_xlabel('Time (s)')
fig.suptitle(f'CASCADE Predictions - {model_name}', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Prediction Statistics

In [ ]:
# Remove NaN padding for stats
valid = spike_rates[~np.isnan(spike_rates)]

print("Prediction Statistics:")
print(f"  Mean spike rate:   {np.mean(valid):.4f}")
print(f"  Median spike rate: {np.median(valid):.4f}")
print(f"  Max spike rate:    {np.max(valid):.4f}")
print(f"  Fraction zero:     {np.mean(valid == 0):.2%}")
print(f"  Non-zero neurons:  {np.sum(np.nanmax(spike_rates, axis=1) > 0)}/{spike_rates.shape[0]}")

# Heatmap of activity
fig, ax = plt.subplots(figsize=(14, 6))
# Sort neurons by total activity for better visualization
activity_order = np.argsort(np.nansum(spike_rates, axis=1))[::-1]
im = ax.imshow(spike_rates[activity_order[:50], :], aspect='auto', cmap='hot',
               extent=[0, traces.shape[1]/frame_rate, 50, 0])
ax.set_xlabel('Time (s)')
ax.set_ylabel('Neuron (sorted by activity)')
ax.set_title('Spike rate heatmap (top 50 most active neurons)')
plt.colorbar(im, ax=ax, label='Spike rate')
plt.tight_layout()
plt.show()

## 6. Model Comparison
Compare different smoothing parameters.